# Turno — Final Decision-Oriented Analytical Checks

**Scope:** M2 recorded-exposure chronology; conditional second completed project within 30 days; optional D1 project-weighted baseline adjustment; independent source checks. No new EDA, eligibility rule, causal retention metric, product recommendation, or retrospective KPI replacement.

All timestamps use **UTC**. The experiment-completion window is `[2025-08-18 00:00, 2025-11-20 00:00)`; 30-day follow-up eligibility requires the exact right endpoint to fall **strictly inside** that completion interval. Captures are observed payment-log timestamps, not proof of full settlement. The new D1 specification is a sensitivity; previous Sprint 1 and inferential outputs remain approved and unchanged.


## 1. Source loading, guarded joins and existing metric consistency

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'final_decision_checks.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'final_decision_checks.py').exists():
    raise FileNotFoundError('Run inside the Turno repository with src/ available.')
sys.path.insert(0, str(ROOT))
from src.experiment_impact import build_project_panel, build_host_panel
from src.inferential_validation import paired_period_host_panel
from src.final_decision_checks import (m2_recorded_exposure,
    second_within_30_days, d1_project_weighted_baseline_sensitivity)

hosts = pd.read_csv(ROOT/'data/raw/experiment_exposures.csv', dtype='string')
assignments = pd.read_csv(ROOT/'data/raw/assignments.csv', dtype='string')
logs = pd.read_csv(ROOT/'data/raw/payment_logs.csv', dtype='string')
project = build_project_panel(hosts, assignments, logs)
assert len(hosts)==14708 and len(assignments)==len(project)==279847
assert len(logs)==306957 and project.assignment_id.is_unique
assert int(project.in_experiment.sum())==201302
assert hosts.experiment_group.value_counts().to_dict()=={'control':7377,'test':7331}
print('PASS: host / project / payment-log reconciliation and safeguarded joins')
print('Registered hosts:',len(hosts),'Experiment completions:',int(project.in_experiment.sum()))

PASS: host / project / payment-log reconciliation and safeguarded joins
Registered hosts: 14708 Experiment completions: 201302


## 2. M2 — Recorded Post-Exposure Chronology Sensitivity

The original per-registered-host M2 must **not** be replaced by this filtered descriptive view. Exposure timestamps do not necessarily identify when the policy was applied; the post-exposure subset can have unequal observation opportunity.

In [2]:
m2, flagged_firsts = m2_recorded_exposure(project, hosts)
assert int(m2.flagged_first.sum()) == 2110
assert int(m2.pre_recorded_exposure.sum())+int(m2.post_or_at_recorded_exposure.sum()) == 2110
raw = 100 * (m2.iloc[1].original_M2_per_host-m2.iloc[0].original_M2_per_host)
post = 100 * (m2.iloc[1].post_recorded_exposure_per_host-m2.iloc[0].post_recorded_exposure_per_host)
print('Original M2 Test − Control: %.6f pp' % raw)
print('Recorded post-exposure chronology sensitivity: %.6f pp' % post)
print('Change versus original: %.6f pp' % (post-raw))
display(m2)

# M3 is previously approved: verify its counts, without new inference.
panel = build_host_panel(hosts, project)
m3_counts = panel.groupby('experiment_group',sort=True).agg(hosts=('host_id','size'),
    hosts_2plus=('two_plus','sum'))
assert m3_counts.loc['control','hosts_2plus']==5957
assert m3_counts.loc['test','hosts_2plus']==5991
print('Approved M3 count reconciliation only:')
display(m3_counts)

Original M2 Test − Control: 1.476750 pp
Recorded post-exposure chronology sensitivity: 1.460898 pp
Change versus original: -0.015852 pp


,group,registered_hosts,flagged_first,pre_recorded_exposure,post_or_at_recorded_exposure,pre_share_of_flagged,post_share_of_flagged,original_M2_per_host,post_recorded_exposure_per_host
0,control,7377,1004,26,978,0.025896,0.974104,0.136099,0.132574
1,test,7331,1106,27,1079,0.024412,0.975588,0.150866,0.147183


Approved M3 count reconciliation only:


,hosts,hosts_2plus
experiment_group,,
control,7377,5957
test,7331,5991


## 3. Second Completion Within 30 Days of First-Ever Completion

**Strict 30-day complete-follow-up cohort.** The first event is a source-flagged first historical completion during the experiment. A second must be a distinct assignment with a **strictly later** completed timestamp and occur on or before the exact +30-day anniversary. Project created timestamps identify whether the subsequently completed project was already reserved; they are not a measure of a newly decided return. These conditional groups are potentially post-treatment selected and **not causally comparable**.

In [3]:
second_summary, second_hosts, second_diagnostics = second_within_30_days(project)
assert int(second_summary.eligible_hosts.sum())==1552
assert int(second_summary.second_within_30d.sum())==1055
assert int(second_summary.second_created_before_first.sum()+second_summary.second_created_simultaneously.sum()+second_summary.second_created_after_first.sum())==1055
print('30-day cohort, right censoring and tie diagnostics:',second_diagnostics)
print('Second-completion rate Test − Control: %.4f pp' % (100*(second_summary.iloc[1].second_rate-second_summary.iloc[0].second_rate)))
display(second_summary)
print('Latest included first historical completion (UTC):',second_hosts.first_completed_at.max())
# Historical first flag quality: preserve observed same-time ambiguity.
first_any = project.loc[project.is_first_true, ['host_id','assignment_id','completed_utc']].rename(
    columns={'assignment_id':'first_assignment_id','completed_utc':'first_completed_at'})
assert first_any.host_id.is_unique
joined = project[['host_id','assignment_id','completed_utc']].merge(first_any,on='host_id',validate='many_to_one')
assert not ((joined.assignment_id!=joined.first_assignment_id)&joined.completed_utc.lt(joined.first_completed_at)).any()
tie_hosts = joined.loc[(joined.assignment_id!=joined.first_assignment_id)&joined.completed_utc.eq(joined.first_completed_at),'host_id'].nunique()
print('Other projects tied with a flagged first completion (all periods):',tie_hosts)

30-day cohort, right censoring and tie diagnostics: {'flagged_experiment_firsts': 2110, 'eligible_firsts': 1552, 'censored_firsts': 558, 'earlier_or_tied_hosts': 1, 'tied_hosts': 1, 'strict_followup_end': '2025-11-20 00:00:00+00:00'}
Second-completion rate Test − Control: 1.8701 pp


,group,eligible_hosts,second_within_30d,second_rate,second_created_before_first,second_created_simultaneously,second_created_after_first,flagged_first_earlier_or_tied_hosts,exactly_tied_second_candidate_hosts
0,control,730,489,0.669863,269,0,220,1,1
1,test,822,566,0.688564,291,0,275,0,0


Latest included first historical completion (UTC): 2025-10-20 23:19:30+00:00


Other projects tied with a flagged first completion (all periods): 2


## 4. Optional D1 — project-weighted baseline adjustment

A **sensitivity**, not an ITT, difference-in-differences, or a correction for post-treatment project selection. Estimate a conditional Test coefficient with one row per registered host, experiment completed-project count as the WLS weight, and experimental captured-D1 fraction as the response. Regress on Test, log(1 + baseline completed projects), . The baseline completion count is strictly pre-experimental; no baseline D1 payment evidence is used, since 24-hour windows on the last baseline day may cross the experiment start. Hosts without baseline projects have genuine zero counts (no imputed baseline rate); hosts without experiment projects have zero weight in this project-conditioned comparison. HC3 robust host-level errors are appropriate because each host enters once. A common linear response slope is an additional modeling assumption.

In [4]:
paired = paired_period_host_panel(hosts, project)
d1 = d1_project_weighted_baseline_sensitivity(paired)
assert d1['hosts_total']==14708 and d1['projects_experiment']==201302
assert d1['hosts_with_zero_experiment']==836
assert abs(100*d1['raw_diff'] - (-0.7302356701006874)) < 0.0001
print('Raw D1 Test − Control: %.4f pp' % (100*d1['raw_diff']))
print('Weighted HC3 adjusted D1 contrast: %.4f pp' % (100*d1['adjusted_diff']))
print('95%% HC3 CI: [%.4f, %.4f] pp' % (100*d1['ci95_low'],100*d1['ci95_high']))
print('Hosts with zero baseline completions:',d1['hosts_without_baseline'])
print('Hosts with zero experiment completions (zero D1 weight):',d1['hosts_with_zero_experiment'])
display(pd.DataFrame([d1]).T.rename(columns={0:'Value'}))

Raw D1 Test − Control: -0.7302 pp
Weighted HC3 adjusted D1 contrast: -0.7645 pp
95% HC3 CI: [-1.1770, -0.3520] pp
Hosts with zero baseline completions: 4424
Hosts with zero experiment completions (zero D1 weight): 836


,Value
raw_diff,-0.007302
adjusted_diff,-0.007645
ci95_low,-0.01177
ci95_high,-0.00352
pvalue,0.000281
hosts_total,14708
hosts_with_experiment,13872
hosts_with_zero_experiment,836
hosts_without_baseline,4424
projects_experiment,201302


## 5. Reuse of previously approved operational contrasts

The incident-CS sensitivity was already completed (D1 −0.730 pp to −0.743 pp upon symmetrically removing 30,460 incident-completed projects). The following previously approved **additional projects without observed capture per 1,000 completed projects** are copied as reference values, not recalculated or claimed to be money lost. D30 has a conservatively complete, smaller follow-up cohort.

In [5]:
approved_operational = pd.DataFrame([
    ('D1',7.30,'Full experiment completion cohort'),
    ('D7',1.63,'Full experiment completion cohort'),
    ('D14',0.97,'Full experiment completion cohort'),
    ('D30',0.77,'Conservative D30 complete-coverage cohort'),
],columns=['horizon','additional_no_observed_capture_per_1000','denominator_note'])
display(approved_operational)
print('Values reproduced from prior approved reports only; no additional test conducted.')

,horizon,additional_no_observed_capture_per_1000,denominator_note
0,D1,7.30,Full experiment completion cohort
1,D7,1.63,Full experiment completion cohort
2,D14,0.97,Full experiment completion cohort
3,D30,0.77,Conservative D30 complete-coverage cohort


Values reproduced from prior approved reports only; no additional test conducted.


## 6. Independent standard-library audit of M2 and 30-day second completion

This check reads the unmodified assignment and exposure CSVs with `csv.DictReader` and Python `datetime`, not the pandas project panel or the new helper. It independently validates the most decision-relevant new counts and cutoff.

In [6]:
import csv
from collections import defaultdict
from datetime import datetime, timezone, timedelta

def parse_utc_text(x):
    return datetime.fromisoformat(x).replace(tzinfo=timezone.utc)

registry={}
with open(ROOT/'data/raw/experiment_exposures.csv',newline='',encoding='utf-8') as f:
    for r in csv.DictReader(f):
        assert r['host_id'] not in registry
        registry[r['host_id']] = (r['experiment_group'],parse_utc_text(r['first_exposed_at']))
period_start=parse_utc_text('2025-08-18 00:00:00')
period_end=parse_utc_text('2025-11-20 00:00:00')
source_by_host=defaultdict(list)
flagged={}
m2_audit=defaultdict(lambda: {'flagged':0,'pre':0,'post':0})
with open(ROOT/'data/raw/assignments.csv',newline='',encoding='utf-8') as f:
    for r in csv.DictReader(f):
        host=r['host_id']
        assert host in registry
        completed=parse_utc_text(r['completed_at']);created=parse_utc_text(r['created_at'])
        source_by_host[host].append((r['assignment_id'],completed,created))
        if r['is_first_completed_project'].lower()=='true' and period_start<=completed<period_end:
            assert host not in flagged
            flagged[host]=(r['assignment_id'],completed)
            group, exposure=registry[host]
            m2_audit[group]['flagged']+=1
            m2_audit[group]['pre' if completed<exposure else 'post']+=1
second_audit=defaultdict(lambda: {'eligible':0,'second':0,'book_before':0,'book_equal':0,'book_after':0})
for host,(first_id,first_at) in flagged.items():
    deadline=first_at+timedelta(days=30)
    if not deadline<period_end:continue
    group,_=registry[host]; second_audit[group]['eligible']+=1
    possibilities=sorted((completed,assignment_id,created) for assignment_id,completed,created in source_by_host[host]
                          if assignment_id!=first_id and first_at<completed<=deadline)
    if not possibilities:continue
    second_audit[group]['second']+=1
    completed,assignment_id,created=possibilities[0]
    second_audit[group]['book_before' if created<first_at else 'book_equal' if created==first_at else 'book_after']+=1
for row in m2.itertuples(index=False):
    audit=m2_audit[row.group]
    assert (row.flagged_first,row.pre_recorded_exposure,row.post_or_at_recorded_exposure)==(audit['flagged'],audit['pre'],audit['post'])
for row in second_summary.itertuples(index=False):
    audit=second_audit[row.group]
    assert (row.eligible_hosts,row.second_within_30d,row.second_created_before_first,
            row.second_created_simultaneously,row.second_created_after_first)==(
            audit['eligible'],audit['second'],audit['book_before'],audit['book_equal'],audit['book_after'])
print('PASS: Independent stdlib audit matched M2 chronological groups and all conditional 30-day second-event/baseline booking categories.')
print('M2 audit:',dict(m2_audit))
print('Second-event audit:',dict(second_audit))

PASS: Independent stdlib audit matched M2 chronological groups and all conditional 30-day second-event/baseline booking categories.
M2 audit: {'test': {'flagged': 1106, 'pre': 27, 'post': 1079}, 'control': {'flagged': 1004, 'pre': 26, 'post': 978}}
Second-event audit: {'test': {'eligible': 822, 'second': 566, 'book_before': 291, 'book_equal': 0, 'book_after': 275}, 'control': {'eligible': 730, 'second': 489, 'book_before': 269, 'book_equal': 0, 'book_after': 220}}


## Interpretation boundaries

- **M2:** the Test-positive difference is essentially unchanged when events before recorded first exposure are excluded for a diagnostic only. The exposure timestamp is not verified assignment or policy activation time.
- **Second project:** continuation is common among the **selected, 30-day-complete first-completion hosts**; many second projects were booked before their first one was completed. No inference about causal retention or uncompleted bookings is possible.
- **D1:** the baseline-adjusted, project-weighted HC3 conditional contrast remains negative. It does not identify a causal impact on a fixed project set. Baseline D1 imbalance, unobserved policy timing, and financial settlement ambiguity remain.
- **Operational risk:** D1/D7/D14/D30 values per 1,000 are contrasts in observed captured-at timestamps, not confirmed monetary losses.
- **Reproduction:** synthetic tests, source checks and notebook execution are documented in `reports/final_decision_checks.md`. Prior notebooks and results are unchanged. This supplement makes no autonomous Ship/Kill/Modified Ship recommendation.
